In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scipy.optimize
from scipy.optimize import minimize
from scipy.stats import norm
from scipy.special import logsumexp
import warnings
from scipy.optimize import curve_fit
from scipy.optimize import approx_fprime
import scipy.stats as stats
from pybads import BADS

from matplotlib.colors import LinearSegmentedColormap

# TEMP
warnings.simplefilter("ignore", category=FutureWarning)

from src.src import *

# Set some defaults
plt.rc("axes.spines", top=False, right=False)
plt.rcParams["mathtext.default"] = "regular"
sns.set_theme(font_scale=1.2)
sns.set_style("ticks")
# sns.set_palette('deep')

%config InlineBackend.figure_format = "retina"

## The Plan

### Parameter and model recovery 
1) Simulate data using best-fit parameters from each model (or samples from within the range of best-fits). For example, 100 simulations of each model within range of best-fit parameters.
2) Refit simulated data using MLE.
3) Correlate parameters used for simulation with recovered parameters.
4) Model comparison with AIC/BIC. 

In [2]:
# Read in vmr data
sdt = pd.read_csv("../behavioural/data/processed-sdt.csv")

# Read in csv with MLEs from all models
# Need a function to convert string back to numpy array
def converter(input_str):
    return np.fromstring(input_str[1:-1], sep=' ')

fits_sdt = pd.read_csv("model-fits/fits_sdt.csv", converters={"params":converter})
fits_sdt.head()


,SN,model,fit_task,nll,aic,bic,params,C,sigma_v,b,sigma_comb,alpha,beta,sigma_p,sigma_pert,aic_weighted,bic_weighted
0,1,vco,sdt,318.538186,643.076372,657.122698,"[3.41236554, 1.82896459, 0.11050354]",3.412366,1.828965,0.110504,NaN,NaN,NaN,NaN,NaN,935.269295,950.448148
1,2,vco,sdt,286.700460,579.400920,593.447246,"[5.10994453, 2.46276086, -2.78588448]",5.109945,2.462761,-2.785884,NaN,NaN,NaN,NaN,NaN,843.107859,858.289288
2,3,vco,sdt,466.611582,939.223163,953.189484,"[3.46888195, 3.89033129, -0.20806376]",3.468882,3.890331,-0.208064,NaN,NaN,NaN,NaN,NaN,1309.749992,1324.719380
3,4,vco,sdt,367.046348,740.092696,754.142779,"[4.10654312, 2.6443675, -0.2592485]",4.106543,2.644368,-0.259249,NaN,NaN,NaN,NaN,NaN,1071.766618,1086.935144
4,5,vco,sdt,266.566258,539.132516,553.178842,"[4.82299976, 2.26539177, -0.07828226]",4.823000,2.265392,-0.078282,NaN,NaN,NaN,NaN,NaN,783.651941,798.830794


In [3]:
# Group by model and stack params into 2d array
params = fits_sdt.groupby("model")["params"].apply(np.stack)
params_vco = params["vco"]
params_co = params["co"]
params_piece = params["piece"]

# Set upper and lower bounds on simulated params
vco_lb = params_vco.min(axis=0)
vco_ub = params_vco.max(axis=0)
co_lb = params_co.min(axis=0)
co_ub = params_co.max(axis=0)
piece_lb = params_piece.min(axis=0)
piece_ub = params_piece.max(axis=0)

# Pull out experimental parameters (same for all 16 subjects)
sdt = sdt.loc[sdt["TN"] > 106, :].reset_index(drop=True)
rotation = sdt.loc[sdt["SN"] == 1, "delta_xv"].values
num_trials = len(np.unique(sdt["TN"]))


In [18]:
# Determine how many simulations to run
n_sims = 1

# We switch off the printing from BADS
options = {"display" : 'off'}

fit_task = "sdt"
n_sims = 1
n_inits = 1

# models = ["vco", "co", "piece"]
models = ["vco"]
simulated_params = []
recovered_params = []
model = []
winner_bic = []
winner_aic = []
iteration = []

# Choose subject to base simultation on (x_f)
subj_id = 1

# Simulate sdt with best-fit parameters
sdt_subj = sdt.loc[(sdt["SN"] == subj_id) & (sdt["session"] == "sdt"), :].reset_index(drop=True)
cols_to_copy = ["SN", "x_f", "session", "delta_xv", "sigma_motor_sdt"]
df_vco = pd.DataFrame()
df_vco[cols_to_copy] = sdt_subj[cols_to_copy]

resp_vco = np.zeros((n_sims, len(sdt_subj)))
resp_co = np.zeros((n_sims, len(sdt_subj)))
resp_piece = np.zeros((n_sims, len(sdt_subj)))

for i in range(n_sims):
    print(i)
    # Generate parameter vals for each model
    vco_simparams = np.random.uniform(low=vco_lb, high=vco_ub)
    co_simparams = np.random.uniform(low=co_lb, high=co_ub)
    piece_simparams = np.random.uniform(low=piece_lb, high=piece_ub)

    for j in range(len(models)):
        print(j)
        if j == 0:
            # VCO model
            resp_vco[i, :] = vco_simulate_sdt(
                df=sdt_subj,
                C=vco_simparams[0], 
                sigma_v=vco_simparams[1], 
                b=vco_simparams[2], 
            )
            simulated_model = "vco"
            simulated_params.append(vco_simparams)
            df_vco["pert_resp"] = resp_vco[i, :]
            df_vco["no_fit"] = False
            
        elif j == 1: 
            # CO model
            resp_co[i, :] = co_simulate_sdt(
                df=sdt_subj,
                C=co_simparams[0], 
                sigma_comb=co_simparams[1],
                b=None
            )
            simulated_model = "co"
            simulated_params.append(co_simparams)
        elif j == 2:
            # PIECE model
            resp_piece[i, :] = piece_simulate_sdt(
                df=sdt_subj,
                alpha=piece_simparams[0], 
                beta=piece_simparams[1],
                sigma_p=piece_simparams[2],
                sigma_pert=piece_simparams[3],
                b=piece_simparams[4],
            )
            simulated_model = "piece"
            simulated_params.append(piece_simparams)
    
        # Fit simulated data:
        # VCO parameter bounds (C, sigma_v, b) 
        bounds       = [(0.1, 10), (0.1, 10.0), (-5, 5)]
        plaus_bounds = [(0.5, 5), (0.5, 6), (-2.5, 2.5)]
        fits_vco, aic_vco, bic_vco = fit_bads("vco", subj_id, df_vco, bounds, plaus_bounds, n_inits=n_inits, 
                                              n_sims=n_sims, fit_task=fit_task, options=options)
        if simulated_model == "vco":
            recovered_params.append(fits_vco.x)

        # # CO parameter bounds (C, sigma_comb) 
        # bounds = [(0.1, 10), (0.1, 10.0)]
        # plaus_bounds = [(0.5, 5), (0.5, 6)]
        # fits_co, aic_co, bic_co = fit_bads("co", subj_id, sdt_subj, bounds, plaus_bounds, n_inits=n_inits, 
        #                              n_sims=n_sims, fit_task=fit_task, options=options)
        # if simulated_model == "co":
        #     recovered_params.append(fits_co.x)

        # # PIECE parameter bounds: alpha, beta, sigma_p, sigma_pert, b,
        # bounds       = ((0.1, 5), (0.001, 1), (0.1, 10), (0.1, 120), (-5, 5))
        # plaus_bounds = ((0.5, 4), (0.01, 0.5), (0.3, 6), (3, 20), (-2.5, 2.5))
        # fits_piece, aic_piece, bic_piece = fit_bads("piece", subj_id, sdt_subj, bounds, plaus_bounds, n_inits=n_inits, 
        #                                 n_sims=n_sims, fit_task=fit_task, options=options)
        # if simulated_model == "piece":
        #     recovered_params.append(fits_piece.x)

        # # Store winning model
        # winner_aic.append(models[np.argmin([aic_vco, aic_co, aic_piece])])
        # winner_bic.append(models[np.argmin([bic_vco, bic_co, bic_piece])])

        # # Update variables
        # model.append(simulated_model)
        # iteration.append(i)

# Store variables in data frame
df_model = pd.DataFrame({
    "iteration": iteration,
    "model":model, 
    "winner_aic":winner_aic, 
    "winner_bic":winner_bic,
    "simulated_params": simulated_params,
    "recovered_params": recovered_params
})
        

0
0
Subject Number: 1
Running optimization 0...


ValueError: All arrays must be of the same length

In [ ]:
resp_vco

In [ ]:
np.zeros(2, 3)